# Notebook 03 — Link MTSamples Clinical Notes to MPI

**Goal:** Assign each MTSamples clinical note a `patient_id` from the MPI.

**Linking logic:**
- Match note's `medical_specialty` to patient's `primary_condition` using a specialty-condition mapping
- Extended mapping for specialties with no direct condition match
- Random fallback only for non-clinical specialties (Letters, Autopsy, IME etc.)

**Output:** `data_preparation/linked/notes_linked.csv`

## 1. Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import random
import os

MTSAMPLES_PATH = "../raw/mtsamples/mtsamples.csv"
MPI_PATH       = "../linked/patients_master.csv"
OUTPUT_DIR     = "../linked/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

Paths OK


## 2. Load Data

In [2]:
mtsamples_raw = pd.read_csv(MTSAMPLES_PATH)
mpi           = pd.read_csv(MPI_PATH)

# Drop unnamed index columns
mtsamples_raw = mtsamples_raw.loc[:, ~mtsamples_raw.columns.str.contains("^Unnamed")]

# Drop rows with missing transcription text
mtsamples = mtsamples_raw.dropna(subset=["transcription"]).copy()
mtsamples = mtsamples.reset_index(drop=True)

print(f"MTSamples raw rows      : {len(mtsamples_raw)}")
print(f"MTSamples after cleaning: {len(mtsamples)}")
print(f"Dropped (no transcript) : {len(mtsamples_raw) - len(mtsamples)}")
print(f"MPI patients            : {len(mpi)}")
print()
print("All specialties:")
print(mtsamples["medical_specialty"].value_counts())

MTSamples raw rows      : 4999
MTSamples after cleaning: 4966
Dropped (no transcript) : 33
MPI patients            : 1163

All specialties:
medical_specialty
Surgery                          1088
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  224
Neurology                         223
SOAP / Chart / Progress Notes     166
Urology                           156
Obstetrics / Gynecology           155
Discharge Summary                 108
ENT - Otolaryngology               96
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    61
Psychiatry / Psychology            53
Office Notes                       50
Podiat

## 3. Define Full Specialty → Condition Mapping

In [3]:
SPECIALTY_CONDITION_MAP = {
    "Cardiovascular / Pulmonary": [
        "heart", "cardiac", "coronary", "hypertension", "atrial",
        "congestive", "hypoxemia", "pulmonary", "asthma", "copd",
        "pneumonia", "bronchitis", "sinusitis"
    ],
    "Orthopedic": [
        "back pain", "neck pain", "osteoporosis", "fracture",
        "sprain", "arthritis", "osteoarthritis", "carpal"
    ],
    "Neurology": [
        "stroke", "epilepsy", "migraine", "anxiety", "depression",
        "dementia", "alzheimer", "neuropathy"
    ],
    "Gastroenterology": [
        "pain", "chronic", "obesity", "diabetes", "anemia",
        "hypertension", "hyperlipidemia", "prediabetes",
        "back pain", "sinusitis", "viral"
    ],
    "General Medicine": [
        "diabetes", "obesity", "prediabetes", "hyperlipidemia",
        "anemia", "hypothyroid", "vitamin"
    ],
    "Obstetrics / Gynecology": [
        "miscarriage", "ovarian", "uterine", "gynecol"
    ],
    "ENT - Otolaryngology": [
        "sinusitis", "otitis", "pharyngitis", "tonsil"
    ],
    "Urology": [
        "kidney", "renal", "urinary", "bladder", "prostate",
        "hypertension", "diabetes"
    ],
    "Hematology - Oncology": [
        "cancer", "tumor", "leukemia", "lymphoma", "anemia"
    ],
    "Dermatology": [
        "skin", "dermatitis", "eczema", "psoriasis", "rash",
        "obesity", "diabetes", "anemia"
    ],
    "Ophthalmology": [
        "diabetes", "hypertension", "stroke", "obesity",
        "hyperlipidemia", "prediabetes", "anemia"
    ],
    "Podiatry": [
        "foot", "ankle", "plantar", "heel", "sprain", "back pain"
    ],
    "Psychiatry / Psychology": [
        "anxiety", "depression", "panic", "bipolar", "schizophrenia"
    ],
    "Endocrinology": [
        "diabetes", "thyroid", "hormone", "adrenal",
        "prediabetes", "hyperlipidemia", "obesity"
    ],
    "Infectious Disease": [
        "covid", "influenza", "viral", "bacterial", "sepsis"
    ],
    "Surgery": [
        "fracture", "cancer", "tumor", "back pain",
        "sprain", "osteoporosis", "obesity", "chronic"
    ],
    "Consult - History and Phy.": [
        "hypertension", "diabetes", "obesity", "anemia",
        "hyperlipidemia", "prediabetes", "chronic"
    ],
    "Radiology": [
        "fracture", "cancer", "stroke", "pulmonary",
        "sinusitis", "back pain", "obesity"
    ],
    "SOAP / Chart / Progress Notes": [
        "hypertension", "diabetes", "obesity", "chronic",
        "back pain", "anemia", "prediabetes"
    ],
    "Discharge Summary": [
        "stroke", "heart", "pneumonia", "cancer",
        "fracture", "congestive", "hypoxemia", "covid"
    ],
    "Neurosurgery": [
        "stroke", "back pain", "neck pain",
        "neuropathy", "fracture", "migraine"
    ],
    "Nephrology": [
        "kidney", "renal", "urinary",
        "hypertension", "diabetes", "anemia"
    ],
    "Emergency Room Reports": [
        "fracture", "sprain", "viral", "covid",
        "stroke", "asthma", "anxiety", "chronic"
    ],
    "Pediatrics - Neonatal": [
        "otitis", "pharyngitis", "viral sinusitis",
        "bronchitis", "asthma", "anemia"
    ],
    "Pain Management": [
        "back pain", "neck pain", "chronic",
        "osteoporosis", "arthritis", "fracture", "sprain"
    ],
    "Office Notes": [
        "hypertension", "diabetes", "obesity",
        "anemia", "hyperlipidemia", "chronic"
    ],
    "Bariatrics": [
        "obesity", "diabetes", "hyperlipidemia",
        "prediabetes", "body mass"
    ],
    "Rheumatology": [
        "arthritis", "osteoporosis", "back pain",
        "neck pain", "chronic"
    ],
    "Chiropractic": [
        "back pain", "neck pain", "sprain", "chronic"
    ],
    "Sleep Medicine": [
        "anxiety", "obesity", "depression", "chronic"
    ],
    "Physical Medicine - Rehab": [
        "back pain", "neck pain", "fracture",
        "sprain", "stroke", "osteoporosis"
    ],
    "Allergy / Immunology": [
        "asthma", "sinusitis", "pharyngitis", "viral", "anemia"
    ],
    "Diets and Nutritions": [
        "obesity", "diabetes", "hyperlipidemia",
        "prediabetes", "body mass", "anemia"
    ],
    "Dentistry": [
        "sinusitis", "pharyngitis", "viral",
        "otitis", "chronic", "hypertension"
    ],
    "Cosmetic / Plastic Surgery": [
        "obesity", "back pain", "fracture",
        "sprain", "cancer", "chronic"
    ],
    "Hospice - Palliative Care": [
        "cancer", "congestive", "stroke", "dementia", "chronic"
    ],
}

# Non-clinical specialties → will use random fallback (acceptable)
RANDOM_FALLBACK_SPECIALTIES = [
    "Letters", "IME-QME-Work Comp etc.", "Speech - Language",
    "Lab Medicine - Pathology", "Autopsy"
]

print(f"Clinical specialty mappings : {len(SPECIALTY_CONDITION_MAP)}")
print(f"Random fallback specialties : {len(RANDOM_FALLBACK_SPECIALTIES)}")

Clinical specialty mappings : 36
Random fallback specialties : 5


## 4. Build Specialty → Patient Pool Lookup

In [4]:
all_patient_ids = mpi["patient_id"].tolist()
specialty_patient_pools = {}

for specialty, keywords in SPECIALTY_CONDITION_MAP.items():
    mask = mpi["primary_condition"].str.lower().apply(
        lambda cond: any(kw in str(cond).lower() for kw in keywords)
    )
    matched = mpi[mask]["patient_id"].tolist()
    # If no match found, use full patient pool
    specialty_patient_pools[specialty] = matched if len(matched) > 0 else all_patient_ids
    print(f"{specialty:<40} → {len(specialty_patient_pools[specialty])} patients")

Cardiovascular / Pulmonary               → 290 patients
Orthopedic                               → 181 patients
Neurology                                → 72 patients
Gastroenterology                         → 553 patients
General Medicine                         → 222 patients
Obstetrics / Gynecology                  → 32 patients
ENT - Otolaryngology                     → 172 patients
Urology                                  → 105 patients
Hematology - Oncology                    → 52 patients
Dermatology                              → 192 patients
Ophthalmology                            → 290 patients
Podiatry                                 → 90 patients
Psychiatry / Psychology                  → 17 patients
Endocrinology                            → 177 patients
Infectious Disease                       → 107 patients
Surgery                                  → 343 patients
Consult - History and Phy.               → 465 patients
Radiology                                → 314 patien

## 5. Link Each Note to a Patient ID

In [5]:
def find_patient_for_note(specialty):
    specialty = str(specialty).strip()

    # Non-clinical → random fallback
    if specialty in RANDOM_FALLBACK_SPECIALTIES:
        return random.choice(all_patient_ids), "random_fallback"

    # Clinical specialty → use pool (already falls back to all_patient_ids if empty)
    if specialty in specialty_patient_pools:
        pool = specialty_patient_pools[specialty]
        match_type = "specialty_match" if pool != all_patient_ids else "random_fallback"
        return random.choice(pool), match_type

    # Unknown specialty → random fallback
    return random.choice(all_patient_ids), "random_fallback"


print("Linking notes to patient IDs...")
results = mtsamples["medical_specialty"].apply(find_patient_for_note)

mtsamples["patient_id"] = results.apply(lambda x: x[0])
mtsamples["match_type"] = results.apply(lambda x: x[1])

print("Done.")
print()
print("Match type distribution:")
print(mtsamples["match_type"].value_counts())

Linking notes to patient IDs...
Done.

Match type distribution:
match_type
specialty_match    4902
random_fallback      64
Name: count, dtype: int64


## 6. Reorder Columns

In [6]:
cols = ["patient_id", "match_type"] + [
    c for c in mtsamples.columns if c not in ["patient_id", "match_type"]
]
mtsamples = mtsamples[cols]

print("Final columns:")
print(mtsamples.columns.tolist())
print()
mtsamples.head(3)

Final columns:
['patient_id', 'match_type', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']



,patient_id,match_type,description,medical_specialty,sample_name,transcription,keywords
0,8a69ae35-07ad-e093-0162-ad02d8e1e070,specialty_match,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,e25be42f-7103-4d46-f926-f9eb6f6381b2,specialty_match,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,97a20cf9-630d-939c-2f50-f13c434aee2f,specialty_match,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."


## 7. Quality Check

In [7]:
print("=== Notes Linking Quality Check ===")
print(f"Total notes              : {len(mtsamples)}")
print(f"Unique patients assigned : {mtsamples['patient_id'].nunique()}")
print(f"Patients with 0 notes    : {len(mpi) - mtsamples['patient_id'].nunique()}")
print()
print("Match type distribution:")
print(mtsamples["match_type"].value_counts())
print()
print("Notes per patient (stats):")
print(mtsamples.groupby("patient_id").size().describe())
print()
print("Remaining random fallback specialties:")
print(mtsamples[mtsamples["match_type"] == "random_fallback"]["medical_specialty"].value_counts())

=== Notes Linking Quality Check ===
Total notes              : 4966
Unique patients assigned : 874
Patients with 0 notes    : 289

Match type distribution:
match_type
specialty_match    4902
random_fallback      64
Name: count, dtype: int64

Notes per patient (stats):
count    874.000000
mean       5.681922
std        3.675181
min        1.000000
25%        3.000000
50%        5.000000
75%        8.000000
max       19.000000
dtype: float64

Remaining random fallback specialties:
medical_specialty
Letters                     23
IME-QME-Work Comp etc.      16
Speech - Language            9
Lab Medicine - Pathology     8
Autopsy                      8
Name: count, dtype: int64


## 8. Save Output

In [8]:
output_path = os.path.join(OUTPUT_DIR, "notes_linked.csv")
mtsamples.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {mtsamples.shape}")

Saved → ../linked/notes_linked.csv
Shape  : (4966, 7)
